# IID Experiments — Single-class & Multi-class (Pavia-U)

**No PCA / No AE** — every detector receives raw 103-D bands.  
Score nets (DSM, LRao) carry a **frozen ZCA whitening first layer**.  
Two sweeps per run: **vs n** (fixed ρ) and **vs ρ** (fixed n).  
Metrics: AUC, partial-AUC (Pfa<0.1), Pd@Pfa.

Detectors — single: **AMF, DSM, LLRao, LRao** | multi: **AMF, GMM-Levin, DSM, LLRao, LRao**  
`LLRao` = **linear** LRao; `LRao` = LRao with the **DSM-MLP** architecture (headline), trained with **validation early stopping** (it is the slow model).

**Multi-seed**: set `seed: [42,43,44,45,46]` in the config → automatic mean±std runs.  
**Runtime**: ≈15–30 min/seed on T4 GPU (full config). Use `QUICK=True` to smoke-test.

In [ ]:
# ── Cell 1: Clone / pull repo + install deps ──────────────────────────────
import os, sys
REPO_URL = 'https://github.com/michaelpiro/final-paper-experiment.git'
LOCAL    = '/content/proj'
BRANCH   = 'iid-unified-experiment'

if os.path.exists(os.path.join(LOCAL, '.git')):
    !git -C {LOCAL} fetch --all -q
    !git -C {LOCAL} checkout {BRANCH} -q
    !git -C {LOCAL} pull origin {BRANCH} -q
else:
    !git clone -b {BRANCH} --depth 1 {REPO_URL} {LOCAL}

!pip install -q scikit-learn scipy tqdm matplotlib pyyaml einops

if LOCAL not in sys.path: sys.path.insert(0, LOCAL)
os.chdir(LOCAL)
print('CWD:', os.getcwd())
!git -C {LOCAL} log --oneline -1

In [ ]:
# ── Cell 2: GPU check ─────────────────────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — CPU only. Enable GPU in Runtime > Change runtime type.')
print(f'PyTorch {torch.__version__}  device={DEVICE}')

In [ ]:
# ── Cell 3: Mount Drive + link pavia-u.mat ────────────────────────────────
import os
RESULTS = '/content/results'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS = '/content/drive/MyDrive/final_paper/iid_results'
    os.makedirs('/content/proj/data', exist_ok=True)
    DST = '/content/proj/data/pavia-u.mat'
    if not os.path.exists(DST):
        for src in [
            '/content/drive/MyDrive/final_paper/pavia-u.mat',
            '/content/drive/MyDrive/final_paper/real_datasets/pavia-u.mat',
        ]:
            if os.path.exists(src):
                os.symlink(src, DST)
                print('Linked dataset from', src); break
        else:
            print('WARNING: pavia-u.mat not found on Drive.')
    else:
        print('Dataset already linked.')
except Exception as e:
    print('Drive not mounted:', repr(e))

os.makedirs(RESULTS, exist_ok=True)
assert os.path.exists('/content/proj/data/pavia-u.mat'), \
    'pavia-u.mat missing! Mount Drive and re-run this cell.'
print('RESULTS dir:', RESULTS)

In [ ]:
# ── Cell 4: Config overrides ───────────────────────────────────────────────
# QUICK=True  → fast smoke-test  (~3 min per seed)
# QUICK=False → use config as-is (full paper run)
QUICK = False

QUICK_OVERRIDES = dict(
    n_train_list    = [100, 500],
    rho_list        = [0.003, 0.01, 0.1],
    n_fixed_for_rho = 100,
    dsm_epochs      = 200,
    lrao_epochs     = 400,
    test_size       = 300,
    seed            = 42,    # single seed for quick test
    device          = DEVICE,
    # MLP LRao + validation early stopping
    run_lrao_mlp         = True,
    lrao_val_fraction    = 0.1,
    lrao_val_check_every = 1,
    lrao_patience        = 3,
    lrao_min_delta       = 0.005,
)
FULL_OVERRIDES = dict(
    device=DEVICE,
    # ---- MLP LRao + LRao validation early stopping (Zschetzsche et al.) ----
    # Naming: LLRao = linear LRao; LRao = the MLP LRao (headline), which reuses the
    # DSM-MLP architecture (auto-detected). Set run_lrao_mlp=False to drop the MLP LRao.
    # Early stopping = monitor val tr(J*) every epoch, stop after `patience` epochs
    # with no meaningful (>min_delta) gain, keep the best-val weights. Paper: patience=3.
    run_lrao_mlp         = True,   # add the MLP LRao (headline 'LRao') alongside LLRao
    lrao_val_fraction    = 0.1,    # 0 disables val early stopping
    lrao_val_check_every = 1,      # check val every epoch (paper); raise to subsample
    lrao_patience        = 3,      # stop after this many checks w/o a meaningful gain
    lrao_min_delta       = 0.005,  # min RELATIVE val tr(J*) gain to reset patience
                                   # (lower → stops later; raise → stops sooner)
)
OVERRIDES = QUICK_OVERRIDES if QUICK else FULL_OVERRIDES
print('Mode:', 'QUICK smoke-test' if QUICK else 'FULL run (config as-is)')
print('Overrides:', OVERRIDES)

In [ ]:
# ── Cell 5: Display helper (PNG saved alongside every PDF) ────────────────
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def show_fig(pdf_path: str, title: str):
    """Display the PNG that iid_core saves alongside every PDF."""
    png_path = pdf_path.replace('.pdf', '.png')
    if os.path.exists(png_path):
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.imshow(mpimg.imread(png_path))
        ax.axis('off')
        ax.set_title(title, fontsize=10)
        plt.tight_layout()
        plt.show()
    elif os.path.exists(pdf_path):
        print(f'  {title}: PDF saved at {pdf_path}  (PNG not found — re-run experiment)')
    else:
        print(f'  {title}: not found')

def show_run_figures(run_dir: str, label: str):
    """Display all standard figures from a run directory."""
    import glob
    fig_dir = os.path.join(run_dir, 'figures')
    FIGS = [
        ('roc_at_nmax.pdf',        'ROC — largest n'),
        ('roc_at_nfixed.pdf',      'ROC — fixed n, sweep ρ'),
        ('auc_vs_n.pdf',           'AUC vs n'),
        ('pauc_vs_n.pdf',          'Partial-AUC (Pfa<0.1) vs n'),
        ('pd_at_fa_vs_n.pdf',      'Pd @ Pfa=0.1 vs n'),
        ('auc_vs_rho.pdf',         'AUC vs ρ'),
        ('pdet_at_pfa_vs_rho.pdf', 'Pd @ Pfa vs ρ'),
        ('roc_auc_bar_nmax.pdf',   'AUC bar chart (multi-seed)'),  # only in agg runs
    ]
    for fname, title in FIGS:
        p = os.path.join(fig_dir, fname)
        if os.path.exists(p) or os.path.exists(p.replace('.pdf','.png')):
            show_fig(p, f'[{label}] {title}')
    for lp in glob.glob(os.path.join(fig_dir, 'loss_curves_*.png')):
        fig, ax = plt.subplots(figsize=(10, 3.5))
        ax.imshow(mpimg.imread(lp)); ax.axis('off')
        ax.set_title(f'[{label}] Training loss curves', fontsize=10)
        plt.tight_layout(); plt.show()

---
## Single-class IID

In [ ]:
# ── Cell S1: Run single-class IID ─────────────────────────────────────────
import yaml, os
from iid_core import run_iid

with open('experiments/iid_single/config.yaml') as f:
    cfg_single = yaml.safe_load(f)
cfg_single.update(OVERRIDES)
cfg_single['results_dir'] = os.path.join(RESULTS, 'iid_single')
os.makedirs(cfg_single['results_dir'], exist_ok=True)

print('=== IID SINGLE-CLASS ===')
print(f"bkg_cls    : {cfg_single.get('bkg_cls')}   target_cls: {cfg_single.get('target_cls')}")
print(f"n_list     : {cfg_single.get('n_train_list')}")
print(f"rho_list   : {cfg_single.get('rho_list', '[default]')}")
print(f"seeds      : {cfg_single.get('seed')}")
print(f"dsm_epochs : {cfg_single.get('dsm_epochs')}   lrao_epochs: {cfg_single.get('lrao_epochs')}")
print(f"device     : {cfg_single.get('device')}")
print()

run_dir_single, metrics_single = run_iid(cfg_single, mode='single')
print('\nSingle run dir:', run_dir_single)

In [ ]:
# ── Cell S2: Display single-class figures ─────────────────────────────────
show_run_figures(run_dir_single, 'single')

In [ ]:
# ── Cell S3: Single-class numeric summary ─────────────────────────────────
import json, pandas as pd

# Works for both single-seed (metrics.json) and multi-seed (metrics_aggregate.json)
mfile = os.path.join(run_dir_single, 'metrics_aggregate.json')
if not os.path.exists(mfile):
    mfile = os.path.join(run_dir_single, 'metrics.json')
with open(mfile) as f:
    m = json.load(f)

dets = list(m['vs_n'].keys())
print('=== vs-n  AUC (single-class) ===')
print(pd.DataFrame({d: m['vs_n'][d]['auc'] for d in dets},
                   index=m['n_list']).rename_axis('n').round(3).to_string())

print('\n=== vs-ρ  AUC (single-class) ===')
print(pd.DataFrame({d: m['vs_rho'][d]['auc'] for d in dets},
                   index=m['rho_list']).rename_axis('rho').round(3).to_string())

roc_key = 'roc_at_nmax'
if roc_key in m:
    print(f"\n=== ROC AUC at n={m['n_list'][-1]} ===")
    for d, v in m[roc_key].items():
        auc_val = v.get('auc') or v.get('auc_mean')  # single vs multi-seed key
        std_val = v.get('auc_std')
        suffix  = f' ± {std_val:.4f}' if std_val is not None else ''
        print(f'  {d:12s}: {auc_val:.4f}{suffix}')

---
## Multi-class IID

In [ ]:
# ── Cell M1: Run multi-class IID ──────────────────────────────────────────
import yaml, os
from iid_core import run_iid

with open('experiments/iid_multi/config.yaml') as f:
    cfg_multi = yaml.safe_load(f)
cfg_multi.update(OVERRIDES)
cfg_multi['results_dir'] = os.path.join(RESULTS, 'iid_multi')
os.makedirs(cfg_multi['results_dir'], exist_ok=True)

print('=== IID MULTI-CLASS ===')
print(f"target_cls : {cfg_multi.get('target_cls')}  bkg = all others (excl {cfg_multi.get('exclude_classes')})")
print(f"n_list     : {cfg_multi.get('n_train_list')}")
print(f"rho_list   : {cfg_multi.get('rho_list', '[default]')}")
print(f"seeds      : {cfg_multi.get('seed')}")
print(f"gmm_K      : {cfg_multi.get('gmm_K')}")
print(f"device     : {cfg_multi.get('device')}")
print()

run_dir_multi, metrics_multi = run_iid(cfg_multi, mode='multi')
print('\nMulti run dir:', run_dir_multi)

In [ ]:
# ── Cell M2: Display multi-class figures ──────────────────────────────────
show_run_figures(run_dir_multi, 'multi')

In [ ]:
# ── Cell M3: Multi-class numeric summary ──────────────────────────────────
import json, pandas as pd

mfile = os.path.join(run_dir_multi, 'metrics_aggregate.json')
if not os.path.exists(mfile):
    mfile = os.path.join(run_dir_multi, 'metrics.json')
with open(mfile) as f:
    mm = json.load(f)

dets_m = list(mm['vs_n'].keys())
print('=== vs-n  AUC (multi-class) ===')
print(pd.DataFrame({d: mm['vs_n'][d]['auc'] for d in dets_m},
                   index=mm['n_list']).rename_axis('n').round(3).to_string())

print('\n=== vs-ρ  AUC (multi-class) ===')
print(pd.DataFrame({d: mm['vs_rho'][d]['auc'] for d in dets_m},
                   index=mm['rho_list']).rename_axis('rho').round(3).to_string())

if 'roc_at_nmax' in mm:
    print(f"\n=== ROC AUC at n={mm['n_list'][-1]} (multi) ===")
    for d, v in mm['roc_at_nmax'].items():
        auc_val = v.get('auc') or v.get('auc_mean')
        std_val = v.get('auc_std')
        suffix  = f' ± {std_val:.4f}' if std_val is not None else ''
        print(f'  {d:12s}: {auc_val:.4f}{suffix}')

---
## Single vs Multi — Side-by-side comparison

In [ ]:
# ── Cell C1: AUC vs n — single vs multi ───────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, matplotlib.ticker as mticker

# Use the SAME canonical color/marker maps as iid_core so every figure (the
# per-run grids and these side-by-side panels) is colour-consistent.
# Naming: LLRao = linear LRao, LRao = MLP LRao (headline).
from iid_core import DETECTOR_COLORS as COLORS, DETECTOR_MARKERS as MARKERS
def _lw(d): return 2.2 if d in ('DSM','LDSM','DSM-lin','DSM-MLP','LRao','LLRao') else 1.4

def _auc_panel(ax, met, title):
    x = np.array(met['n_list'], dtype=float)
    for d in met['vs_n']:
        mu = np.array(met['vs_n'][d]['auc'], dtype=float)
        sd = met['vs_n'][d].get('auc_std')
        c  = COLORS.get(d, '#444444')
        ax.plot(x, mu, color=c, lw=_lw(d), marker=MARKERS.get(d,'o'), label=d)
        if sd: ax.fill_between(x, mu-np.array(sd), mu+np.array(sd), alpha=0.15, color=c)
    ax.set_xscale('log')
    ax.set_xticks(x)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:g}'))
    ax.xaxis.set_minor_locator(mticker.NullLocator())
    ax.tick_params(axis='x', labelsize=8, rotation=45)
    ax.set_xlabel('n'); ax.set_ylabel('AUC')
    ax.set_title(title, fontsize=9); ax.grid(alpha=0.25)
    ax.legend(fontsize=7, loc='lower right')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
_auc_panel(ax1, metrics_single, 'AUC vs n  [single-class]')
_auc_panel(ax2, metrics_multi,  'AUC vs n  [multi-class]')
fig.suptitle('AUC vs training size', fontsize=10)
fig.tight_layout()
out = os.path.join(RESULTS, 'compare_auc_vs_n.pdf')
fig.savefig(out, bbox_inches='tight')
fig.savefig(out.replace('.pdf','.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell C2: AUC vs ρ — single vs multi ───────────────────────────────────
# Reuses COLORS / MARKERS / _lw defined in Cell C1 (canonical maps from iid_core).
import numpy as np, matplotlib.pyplot as plt, matplotlib.ticker as mticker

def _rho_panel(ax, met, title):
    x = np.array(met['rho_list'], dtype=float)
    for d in met['vs_rho']:
        mu = np.array(met['vs_rho'][d]['auc'], dtype=float)
        sd = met['vs_rho'][d].get('auc_std')
        c  = COLORS.get(d, '#444444')
        ax.plot(x, mu, color=c, lw=_lw(d), marker=MARKERS.get(d,'o'), label=d)
        if sd: ax.fill_between(x, mu-np.array(sd), mu+np.array(sd), alpha=0.15, color=c)
    ax.set_xscale('log')
    ax.set_xticks(x)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:g}'))
    ax.xaxis.set_minor_locator(mticker.NullLocator())
    ax.tick_params(axis='x', labelsize=8, rotation=45)
    ax.set_xlabel(r'$\rho$'); ax.set_ylabel('AUC')
    ax.set_title(title, fontsize=9); ax.grid(alpha=0.25)
    ax.legend(fontsize=7, loc='lower right')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
_rho_panel(ax1, metrics_single, f'AUC vs ρ  [single, n={metrics_single["n_fixed"]}]')
_rho_panel(ax2, metrics_multi,  f'AUC vs ρ  [multi,  n={metrics_multi["n_fixed"]}]')
fig.suptitle(r'AUC vs DSM noise level $\rho$', fontsize=10)
fig.tight_layout()
out = os.path.join(RESULTS, 'compare_auc_vs_rho.pdf')
fig.savefig(out, bbox_inches='tight')
fig.savefig(out.replace('.pdf','.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell D1: Download all figures as a zip ────────────────────────────────
import zipfile, glob, os
from google.colab import files

ZIP_PATH = '/content/iid_figures.zip'
to_zip = []
for rd in [run_dir_single, run_dir_multi]:
    for ext in ['*.pdf', '*.png']:
        to_zip.extend(glob.glob(os.path.join(rd, 'figures', ext)))
for cmp in ['compare_auc_vs_n.pdf', 'compare_auc_vs_n.png',
            'compare_auc_vs_rho.pdf', 'compare_auc_vs_rho.png']:
    p = os.path.join(RESULTS, cmp)
    if os.path.exists(p): to_zip.append(p)

base = os.path.dirname(run_dir_single.rstrip('/'))
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in to_zip:
        z.write(p, os.path.relpath(p, base))

print(f'Zipped {len(to_zip)} files → {ZIP_PATH}')
files.download(ZIP_PATH)